In [3]:
import sys
import os

# Add project root to path
sys.path.append(os.path.abspath(".."))

import pandas as pd

from src.cleaning import clean_transactions
from src.fuzzy_grouping import group_merchants

from src.eda import (
    spending_by_category,
    average_monthly_spending_by_category,
    spending_by_merchant,
    average_monthly_spending_by_merchant,
    monthly_spending,
    average_monthly_spending
)

from src.visualization import (
    plot_category_spending,
    plot_avg_monthly_category_spending,
    plot_top_merchants,
    plot_avg_monthly_merchants,
    plot_monthly_spending
)

In [4]:
df = clean_transactions("../data/Discover_Transaction_History.csv")

print(f"Rows loaded: {len(df):,}")
print(f"Unique descriptions: {df['description'].nunique():,}")

/Users/mafphd/personalprojects/src/cleaning.py:33: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["trans_date"] = pd.to_datetime(df["trans_date"], errors="coerce")
/Users/mafphd/personalprojects/src/cleaning.py:36: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["post_date"] = pd.to_datetime(df["post_date"], errors="coerce")


ValueError: could not convert string to float: '(72.63)'

In [15]:
df.head()

,trans._date,post_date,description,amount,category,desc_clean
0,10/17/23,2023-10-19,WALGREENS #10196 WAUWATOSA WI,13.70,Merchandise,WALGREENS WAUWATOSA
1,10/18/23,2023-10-19,5-365 FOOD SERVICE TROY MI,3.05,Restaurants,FOOD SERVICE TROY MI
2,10/18/23,2023-10-19,QUALITY HEATING AND SHEE BROOKFIELD WI,79.65,Services,QUALITY HEATING AND SHEE BROOKFIELD
3,10/18/23,2023-10-19,SOMMER'S INC. MEQUON WI649149,114.59,Automotive,SOMMERS INC MEQUON
4,10/19/23,2023-10-19,MILTOWN EATS LLC 4144349799 WI,30.00,Supermarkets,MILTOWN EATS


In [ ]:
df = group_merchants(
    df,
    threshold=85,
    max_words=2
)

print(f"Unique merchant groups: {df['merchant_clean'].nunique():,}")

In [ ]:
(
    df[[
        "description",
        "desc_clean",
        "merchant_clean"
    ]]
    .sort_values("merchant_clean")
    .head(50)
)

In [ ]:
category_totals = spending_by_category(df)

category_totals.head(20)

In [ ]:
merchant_totals = spending_by_merchant(df)

merchant_totals.head(25)

In [ ]:
merchant_avg_monthly = average_monthly_spending_by_merchant(df)

merchant_avg_monthly.head(25)

In [ ]:
monthly_totals = monthly_spending(df)

monthly_totals

In [ ]:
overall_monthly_average = average_monthly_spending(df)

print(
    f"Average Monthly Spending: "
    f"${overall_monthly_average:,.2f}"
)

In [ ]:
plot_category_spending(
    category_totals
)

In [ ]:
plot_top_merchants(
    merchant_totals,
    top_n=20
)

In [ ]:
plot_monthly_spending(
    monthly_totals
)

In [ ]:
merchant_summary = (
    df.groupby("merchant_clean")
      .agg(
          transaction_count=("amount", "count"),
          total_spend=("amount", "sum"),
          average_transaction=("amount", "mean")
      )
      .reset_index()
      .sort_values(
          "total_spend",
          ascending=False
      )
)

merchant_summary.head(25)

In [ ]:
with pd.ExcelWriter(
    "../outputs/spending_summary.xlsx",
    engine="openpyxl"
) as writer:

    category_totals.to_excel(
        writer,
        sheet_name="Category Totals",
        index=False
    )

    category_avg_monthly.to_excel(
        writer,
        sheet_name="Category Monthly Avg",
        index=False
    )

    merchant_totals.to_excel(
        writer,
        sheet_name="Merchant Totals",
        index=False
    )

    merchant_avg_monthly.to_excel(
        writer,
        sheet_name="Merchant Monthly Avg",
        index=False
    )

    monthly_totals.to_excel(
        writer,
        sheet_name="Monthly Spending",
        index=False
    )

print("Results exported.")